In [4]:

LAST_NAME = "Acapulco Michael"
STUDENT_ID = "TUPM-26-1875"

id_digits = "".join(d for d in STUDENT_ID if d.isdigit())

assert id_digits, "Student ID must contain digits."

SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
NAME_LENGTH = len(LAST_NAME)

RANDOM_SEED = int(id_digits) % (2**32)

verification_value = ID_SUM * NAME_LENGTH + SEED_DIGIT

print("Student:", LAST_NAME)
print("Student ID:", STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Random Seed:", RANDOM_SEED)
print("Verification Value:", verification_value)


Student: Acapulco Michael
Student ID: TUPM-26-1875
Seed Digit: 5
ID Sum: 29
Random Seed: 261875
Verification Value: 469


In [5]:

import numpy as np
import pandas as pd
from pathlib import Path

WORK_DIR = Path.cwd()
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"

measurement_path = DATA_DIR / "secom.data"

OUTPUT_DIR.mkdir(exist_ok=True)

raw_measurements = np.genfromtxt(measurement_path, dtype=float)

missing = np.isnan(raw_measurements)
observed = (~missing).sum(axis=0)

feature_min = np.min(
    np.where(missing, np.inf, raw_measurements), axis=0
)

feature_max = np.max(
    np.where(missing, -np.inf, raw_measurements), axis=0
)

usable_mask = (
    (observed >= 2) &
    (feature_min != feature_max)
)

usable_indices = np.flatnonzero(usable_mask)

rng = np.random.default_rng(RANDOM_SEED)

n_select = 5 + (SEED_DIGIT % 3)

if len(usable_indices) < n_select:
    raise ValueError("Not enough usable features.")

selected_indices = rng.choice(
    usable_indices,
    size=n_select,
    replace=False
)

selected_data = raw_measurements[:, selected_indices]

feature_means = np.nanmean(selected_data, axis=0)
feature_medians = np.nanmedian(selected_data, axis=0)
feature_sds = np.nanstd(selected_data, axis=0, ddof=1)

missing_percentages = (
    np.isnan(selected_data).mean(axis=0) * 100
)

medians = np.nanmedian(selected_data, axis=0)

absolute_deviations = np.abs(
    selected_data - medians
)

mad = np.nanmedian(absolute_deviations, axis=0)

deviation = selected_data - medians

robust_z = np.zeros_like(selected_data)

np.divide(
    0.6745 * deviation,
    mad,
    out=robust_z,
    where=(mad != 0)
)

zero_mad = mad == 0

robust_z[:, zero_mad] = np.where(
    deviation[:, zero_mad] != 0,
    np.inf,
    0
)

robust_z[np.isnan(selected_data)] = np.nan

flagged = np.abs(robust_z) > 3.5

flagged_counts = np.sum(flagged, axis=0)

observed_counts = np.sum(
    ~np.isnan(selected_data), axis=0
)

flagged_percentages = (
    flagged_counts / observed_counts * 100
)

summary = pd.DataFrame({
    "Feature": [
        f"Feature_{i:03d}" for i in selected_indices
    ],
    "Mean": feature_means,
    "Median": feature_medians,
    "Sample_SD": feature_sds,
    "Missing_Percentage": missing_percentages,
    "MAD": mad,
    "Flagged_Count": flagged_counts,
    "Flagged_Percentage": flagged_percentages
})

summary = summary.sort_values(
    by="Flagged_Percentage",
    ascending=False
).reset_index(drop=True)

output_path = OUTPUT_DIR / "measurement_quality.csv"

summary.to_csv(output_path, index=False)

print("Selected Features:", selected_indices)
print("\nMeasurement Quality Summary:")
print(summary.round(4))

print("\nTotal Flagged Observations:", flagged_counts.sum())

print("\nCSV Created:", output_path)


Selected Features: [407 206 308 581 249  20 134]

Measurement Quality Summary:
       Feature     Mean   Median  Sample_SD  Missing_Percentage      MAD  \
0  Feature_581  97.9344  72.2889    87.5210             60.5616  31.1830   
1  Feature_249   0.0011   0.0000     0.0158              0.0000   0.0000   
2  Feature_407   1.2320   1.2397     0.3647              0.5105   0.2109   
3  Feature_020   1.4051   1.4060     0.0167              0.0000   0.0092   
4  Feature_134  39.3920  38.9026     2.9905              0.5105   1.7856   
5  Feature_308   0.2184   0.2194     0.0336              0.0638   0.0204   
6  Feature_206   0.0013   0.0000     0.0506              0.3829   0.0000   

   Flagged_Count  Flagged_Percentage  
0             39              6.3107  
1             22              1.4040  
2             15              0.9622  
3             14              0.8934  
4              5              0.3207  
5              4              0.2554  
6              1              0.0641  


In [ ]:
import numpy as np
import pandas as pd

raw_measurements = np.genfromtxt(
    DATA_DIR / "secom.data",
    dtype=float
)

labels = np.loadtxt(
    DATA_DIR / "secom_labels.data",
    usecols=0,
    dtype=int
)

assert len(raw_measurements) == len(labels)

missing = np.isnan(raw_measurements)
observed = (~missing).sum(axis=0)

feature_min = np.min(
    np.where(missing, np.inf, raw_measurements),
    axis=0
)

feature_max = np.max(
    np.where(missing, -np.inf, raw_measurements),
    axis=0
)

usable_mask = (
    (observed >= 2) &
    (feature_min != feature_max)
)

usable_indices = np.flatnonzero(usable_mask)

feature_names = [
    f"Feature_{i:03d}" for i in range(raw_measurements.shape[1])
]

df = pd.DataFrame(
    raw_measurements,
    columns=feature_names
)

df["Result"] = labels
df["Outcome"] = np.where(labels == -1, "Pass", "Fail")

df.index.name = "Example_ID"

pass_count = (labels == -1).sum()
fail_count = (labels == 1).sum()

sample_size = min(
    200 + ID_SUM,
    pass_count,
    fail_count
)

rng = np.random.default_rng(RANDOM_SEED)

pass_indices = rng.choice(
    np.flatnonzero(labels == -1),
    size=sample_size,
    replace=False
)

fail_indices = rng.choice(
    np.flatnonzero(labels == 1),
    size=sample_size,
    replace=False
)

sample_indices = np.concatenate([
    pass_indices,
    fail_indices
])

sample = df.iloc[sample_indices].copy()

selected_indices = rng.choice(
    usable_indices,
    size=5,
    replace=False
)

selected_features = [
    feature_names[i] for i in selected_indices
]

medians = df[selected_features].median()

df[selected_features] = df[selected_features].fillna(medians)

sample = df.iloc[sample_indices].copy()

grouped_stats = sample.groupby("Outcome")[selected_features].agg(
    ["count", "mean", "median", "std"]
)

comparison = []

for feature in selected_features:

    pass_data = sample.loc[
        sample["Outcome"] == "Pass", feature
    ]

    fail_data = sample.loc[
        sample["Outcome"] == "Fail", feature
    ]

    pass_mean = pass_data.mean()
    fail_mean = fail_data.mean()

    pass_sd = pass_data.std(ddof=1)
    fail_sd = fail_data.std(ddof=1)

    n_pass = len(pass_data)
    n_fail = len(fail_data)

    mean_difference = abs(fail_mean - pass_mean)

    pooled_sd = np.sqrt(
        (
            (n_pass - 1) * pass_sd**2 +
            (n_fail - 1) * fail_sd**2
        ) / (n_pass + n_fail - 2)
    )

    standardized_difference = (
        mean_difference / pooled_sd
        if pooled_sd > 0 else np.nan
    )

    comparison.append({
        "Feature": feature,
        "Pass_Count": n_pass,
        "Fail_Count": n_fail,
        "Pass_Mean": pass_mean,
        "Fail_Mean": fail_mean,
        "Pass_Median": pass_data.median(),
        "Fail_Median": fail_data.median(),
        "Pass_SD": pass_sd,
        "Fail_SD": fail_sd,
        "Absolute_Mean_Difference": mean_difference,
        "Pooled_SD": pooled_sd,
        "Absolute_Standardized_Mean_Difference": standardized_difference
    })

comparison_df = pd.DataFrame(comparison)

comparison_df = comparison_df.sort_values(
    by="Absolute_Standardized_Mean_Difference",
    ascending=False,
    na_position="last"
).reset_index(drop=True)

sample_path = OUTPUT_DIR / "balanced_sample.csv"
comparison_path = OUTPUT_DIR / "feature_comparison.csv"

sample.to_csv(sample_path)
comparison_df.to_csv(comparison_path, index=False)

print("Balanced Sample Size:", len(sample))
print("Pass Records:", (sample["Outcome"] == "Pass").sum())
print("Fail Records:", (sample["Outcome"] == "Fail").sum())

print("\nSelected Features:")
print(selected_features)

print("\nGrouped Statistics:")
print(grouped_stats.round(4))

print("\nFeature Comparison:")
print(comparison_df.round(4))

print("\nHighest-Ranked Feature:")
print(comparison_df.iloc[0]["Feature"])

print("\nAbsolute Standardized Mean Difference:")
print(comparison_df.iloc[0]["Absolute_Standardized_Mean_Difference"])

print("\nNumPy-Pandas Statistics Match:",
      np.allclose(
          sample[selected_features].groupby(sample["Outcome"]).mean().loc["Pass"].values,
          sample[sample["Outcome"] == "Pass"][selected_features].mean().values
      )
)

print("\nGenerated Files:")
print(sample_path)
print(comparison_path)


Balanced Sample Size: 208
Pass Records: 104
Fail Records: 104

Selected Features:
['Feature_182', 'Feature_038', 'Feature_185', 'Feature_467', 'Feature_541']

Grouped Statistics:
        Feature_182                         Feature_038                    \
              count     mean median     std       count     mean   median   
Outcome                                                                     
Fail            104  10.2925  9.985  3.0654         104  86.9247  86.8438   
Pass            104  10.9047  9.875  4.8700         104  86.8038  86.7646   

                Feature_185                        Feature_467          \
            std       count    mean median     std       count    mean   
Outcome                                                                  
Fail     0.6863         104  7.1232   6.52  2.5125         104  7.4277   
Pass     0.3661         104  6.7447   6.37  2.6479         104  5.4262   

                         Feature_541                          
 

In [7]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA_DIR = Path.cwd() / "data"
OUTPUT_DIR = Path.cwd() / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

raw_measurements = np.genfromtxt(
    DATA_DIR / "secom.data",
    dtype=float
)

labels = np.loadtxt(
    DATA_DIR / "secom_labels.data",
    usecols=0,
    dtype=int
)

assert len(raw_measurements) == len(labels)

pass_mask = labels == -1
fail_mask = labels == 1

pass_data = raw_measurements[pass_mask]
fail_data = raw_measurements[fail_mask]

pass_observed = np.sum(~np.isnan(pass_data), axis=0)
fail_observed = np.sum(~np.isnan(fail_data), axis=0)

usable_mask = (
    (pass_observed >= 10) &
    (fail_observed >= 10)
)

usable_indices = np.flatnonzero(usable_mask)

rng = np.random.default_rng(RANDOM_SEED)

selected_indices = rng.choice(
    usable_indices,
    size=5,
    replace=False
)

selected_features = [
    f"Feature_{i:03d}" for i in selected_indices
]

bootstrap_repetitions = 200 + 10 * SEED_DIGIT

def run_bootstrap(seed):

    rng = np.random.default_rng(seed)
    results = []

    for feature_index, feature_name in zip(
        selected_indices, selected_features
    ):

        pass_values = pass_data[:, feature_index]
        fail_values = fail_data[:, feature_index]

        available_pass = pass_values[
            ~np.isnan(pass_values)
        ]

        available_fail = fail_values[
            ~np.isnan(fail_values)
        ]

        full_median = np.nanmedian(
            raw_measurements[:, feature_index]
        )

        imputed_pass = np.where(
            np.isnan(pass_values),
            full_median,
            pass_values
        )

        imputed_fail = np.where(
            np.isnan(fail_values),
            full_median,
            fail_values
        )

        strategies = {
            "Available-Case": (
                available_pass,
                available_fail
            ),
            "Median-Imputation": (
                imputed_pass,
                imputed_fail
            )
        }

        for strategy, (pass_values_used, fail_values_used) in strategies.items():

            mean_difference = (
                np.mean(fail_values_used) -
                np.mean(pass_values_used)
            )

            bootstrap_differences = np.empty(
                bootstrap_repetitions
            )

            for b in range(bootstrap_repetitions):

                pass_sample = rng.choice(
                    pass_values_used,
                    size=len(pass_values_used),
                    replace=True
                )

                fail_sample = rng.choice(
                    fail_values_used,
                    size=len(fail_values_used),
                    replace=True
                )

                bootstrap_differences[b] = (
                    np.mean(fail_sample) -
                    np.mean(pass_sample)
                )

            ci_lower, ci_upper = np.percentile(
                bootstrap_differences,
                [2.5, 97.5]
            )

            results.append({
                "Strategy": strategy,
                "Feature": feature_name,
                "Mean_Difference": mean_difference,
                "Bootstrap_Mean_Difference": np.mean(
                    bootstrap_differences
                ),
                "CI_Lower": ci_lower,
                "CI_Upper": ci_upper,
                "Direction": np.sign(mean_difference),
                "CI_Excludes_Zero": (
                    ci_lower > 0 or ci_upper < 0
                )
            })

    results_df = pd.DataFrame(results)

    results_df["Rank"] = (
        results_df.groupby("Strategy")[
            "Mean_Difference"
        ].rank(
            method="min",
            ascending=False,
            na_option="bottom"
        )
    )

    return results_df.sort_values(
        ["Strategy", "Rank"]
    ).reset_index(drop=True)


results = run_bootstrap(RANDOM_SEED)

verification = run_bootstrap(RANDOM_SEED)

reproducibility_verified = results.equals(verification)

available = results[
    results["Strategy"] == "Available-Case"
].set_index("Feature")

imputed = results[
    results["Strategy"] == "Median-Imputation"
].set_index("Feature")

comparison = pd.DataFrame({
    "Available_Case_Difference": available["Mean_Difference"],
    "Median_Imputation_Difference": imputed["Mean_Difference"],
    "Available_Case_CI_Lower": available["CI_Lower"],
    "Available_Case_CI_Upper": available["CI_Upper"],
    "Median_Imputation_CI_Lower": imputed["CI_Lower"],
    "Median_Imputation_CI_Upper": imputed["CI_Upper"],
    "Available_Case_Rank": available["Rank"],
    "Median_Imputation_Rank": imputed["Rank"]
})

comparison["Absolute_Mean_Difference_Change"] = np.abs(
    comparison["Available_Case_Difference"] -
    comparison["Median_Imputation_Difference"]
)

comparison["Difference_Sign_Changed"] = (
    np.sign(comparison["Available_Case_Difference"]) !=
    np.sign(comparison["Median_Imputation_Difference"])
)

comparison["Ranking_Changed"] = (
    comparison["Available_Case_Rank"] !=
    comparison["Median_Imputation_Rank"]
)

comparison = comparison.reset_index()

comparison_path = (
    OUTPUT_DIR / "robustness_comparison.csv"
)

report_path = (
    OUTPUT_DIR / "sensitivity_report.txt"
)

comparison.to_csv(
    comparison_path,
    index=False
)

largest_change = comparison.loc[
    comparison["Absolute_Mean_Difference_Change"].idxmax()
]

changed_sign = comparison[
    comparison["Difference_Sign_Changed"]
]

changed_ranking = comparison[
    comparison["Ranking_Changed"]
]

report = (
    "STATISTICAL ROBUSTNESS AND SENSITIVITY ANALYSIS\n\n"
    f"Student: {LAST_NAME}\n"
    f"Student ID: {STUDENT_ID}\n"
    f"Random Seed: {RANDOM_SEED}\n"
    f"Selected Features: {', '.join(selected_features)}\n"
    f"Bootstrap Repetitions: {bootstrap_repetitions}\n\n"
    f"Largest Change in Mean Difference: "
    f"{largest_change['Feature']}\n"
    f"Change Value: "
    f"{largest_change['Absolute_Mean_Difference_Change']:.4f}\n\n"
    f"Features with Changed Difference Sign: "
    f"{len(changed_sign)}\n"
    f"Features with Changed Ranking: "
    f"{len(changed_ranking)}\n\n"
    f"Reproducibility Verified: "
    f"{reproducibility_verified}\n\n"
    "INTERPRETATION:\n"
    "Available-case analysis excludes missing measurements, "
    "while median imputation replaces missing values using "
    "the full-dataset median.\n"
    "Differences in bootstrap confidence intervals and feature "
    "rankings indicate sensitivity to data preparation.\n"
)

report_path.write_text(
    report,
    encoding="utf-8"
)

print("Selected Features:", selected_features)
print("Bootstrap Repetitions:", bootstrap_repetitions)

print("\nRobustness Comparison:")
print(comparison.round(4))

print("\nLargest Change in Mean Difference:")
print(largest_change)

print("\nFeatures with Changed Difference Sign:")
print(changed_sign["Feature"].tolist())

print("\nFeatures with Changed Ranking:")
print(changed_ranking["Feature"].tolist())

print("\nReproducibility Verified:", reproducibility_verified)

print("\nGenerated Files:")
print(comparison_path)
print(report_path)

Selected Features: ['Feature_272', 'Feature_582', 'Feature_160', 'Feature_413', 'Feature_236']
Bootstrap Repetitions: 250

Robustness Comparison:
       Feature  Available_Case_Difference  Median_Imputation_Difference  \
0  Feature_160                   208.7934                      205.8604   
1  Feature_272                     2.2201                        2.1925   
2  Feature_582                     0.0006                        0.0006   
3  Feature_236                     0.0000                        0.0000   
4  Feature_413                    -1.2073                       -1.1553   

   Available_Case_CI_Lower  Available_Case_CI_Upper  \
0                  56.7297                 368.9178   
1                  -0.7376                   5.6801   
2                  -0.0001                   0.0014   
3                   0.0000                   0.0000   
4                  -5.4060                   4.1294   

   Median_Imputation_CI_Lower  Median_Imputation_CI_Upper  \
0          